# 📄 NB2 — Map-Reduce Document Summarizer
## The Art of Summarizing Anything, No Matter How Long

LLMs have context limits. GPT-4 can read ~128k tokens. A 300-page book is ~150k tokens. A 3-hour meeting transcript can be 200k+ tokens. What do you do when your document is bigger than what the model can read at once?

**Map-Reduce.** This notebook builds it from scratch.

---

> ⚠️ **LOCAL ONLY** — This notebook uses **Ollama** running on your machine.
> It will **NOT work in Google Colab**. Make sure Ollama is running before you start.

```bash
# Run these once in your terminal (NOT inside this notebook)
ollama pull qwen2.5:7b        # download the model (~4.7 GB)
ollama serve                  # start the Ollama server (if not already running)
```

---

### What you'll learn
- Why context windows are the fundamental constraint in real-world LLM applications
- How to split a long document into overlapping chunks (and why overlap matters)
- How to summarize each chunk independently — the **MAP** step
- How to merge all chunk summaries into one coherent final summary — the **REDUCE** step
- Real-world applications: contracts, research papers, earnings calls, meeting transcripts

### Notebook structure
| Section | What you do |
|---|---|
| **SECTION 1 — CONCEPT** | Watch a fully working pipeline run |
| **SECTION 2 — GUIDED BUILD** | Fill in `___` TODOs to build the pipeline yourself |
| **SECTION 3 — EXPERIMENT** | Change parameters and observe what breaks / improves |
| **SECTION 4 — CHALLENGE** | Three harder extensions (parallel, hierarchical, styled) |

In [ ]:
# Install the two packages we need
# openai  → gives us a Python client that speaks to Ollama's OpenAI-compatible API
# tiktoken → lets us count tokens so we understand when chunking is necessary
!pip install openai tiktoken -q

---
## 🧠 SECTION 1 — CONCEPT: The Context Window Problem

### The 20-pages-at-a-time analogy

Imagine you need to summarize a 500-page book, but you can only read **20 pages at a time** (that's your context window). Here's what you'd do:

1. Read pages 1–20 → write a summary → set aside
2. Read pages 21–40 → write a summary → set aside
3. ... repeat for every section ...
4. Read all the mini-summaries → write **one final summary**

That's **Map-Reduce**. Step 1–3 is the **MAP** (apply the same operation to every piece). Step 4 is the **REDUCE** (combine results into one).

### The full pipeline

```
Original document (10,000+ words)
         │
         ▼  SPLIT
    ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐
    │Chunk1│ │Chunk2│ │Chunk3│ │Chunk4│   ← each fits in the context window
    └──────┘ └──────┘ └──────┘ └──────┘
         │
         ▼  MAP  (summarize each — can run in parallel)
    ┌────┐   ┌────┐   ┌────┐   ┌────┐
    │Sum1│   │Sum2│   │Sum3│   │Sum4│
    └────┘   └────┘   └────┘   └────┘
         │
         ▼  REDUCE  (merge all summaries into one)
    ┌─────────────────────┐
    │   Final Summary     │
    └─────────────────────┘
```

### Why not just truncate the document?

Truncation loses the end of the document — fine for short intros, terrible for contracts or papers where the conclusion matters. Map-Reduce reads **everything**.

### Token counting

Models don't count characters — they count **tokens** (roughly 3–4 characters each in English). A 10,000-word document is about 13,000 tokens. If your model's context window is 32,768 tokens, you might be okay. If it's 100,000 tokens, you definitely need chunking. We'll measure this before deciding.

In [ ]:
from openai import OpenAI  # openai client — works with Ollama's compatible API
import tiktoken             # token counter
import time                 # for measuring how long each step takes

# ─────────────────────────────────────────────────────────────
# Connect to Ollama
# Ollama runs a local HTTP server on port 11434 that speaks the
# same API protocol as OpenAI — so we can reuse the openai client.
# ─────────────────────────────────────────────────────────────
client = OpenAI(
    base_url="http://localhost:11434/v1",  # ⚠️ Ollama must be running locally
    api_key="ollama",                      # Ollama ignores the key, but the client requires one
)
MODEL = "qwen2.5:7b"  # change to whichever model you've pulled (e.g. llama3.2:3b)

# ─────────────────────────────────────────────────────────────
# Sample text — a long overview of modern AI
# In a real project this could be a 100-page PDF or a 3-hour transcript.
# We use a plain string here so the notebook is self-contained.
# ─────────────────────────────────────────────────────────────
SAMPLE_TEXT = """
Artificial intelligence has undergone a dramatic transformation over the past decade.
The introduction of the transformer architecture in 2017 marked a turning point.
Prior to transformers, recurrent neural networks dominated sequence modeling.
These architectures processed sequences token by token, making parallelization difficult.

The transformer addressed these limitations through the self-attention mechanism,
which allows every token to attend to every other token simultaneously.
This parallel computation enabled training on much larger datasets.

BERT in 2018 demonstrated that pre-training followed by fine-tuning could achieve
state-of-the-art results. GPT-2 showed autoregressive models could generate coherent text.
GPT-3 with 175 billion parameters showed few-shot learning from just examples in the prompt.

Instruction tuning and RLHF made these models usable by non-experts.
ChatGPT triggered the current wave of AI application development.
Retrieval-augmented generation emerged as a solution to the knowledge cutoff problem.
Vector databases became essential infrastructure for storing document embeddings.

AI agents represent the next frontier — models that can decompose complex tasks,
call external tools, and iteratively refine their outputs.
LangChain, LangGraph, and CrewAI provide abstractions for building these systems.
Multi-agent architectures show promise for tasks too complex for a single model.

Multimodal models process text, images, audio, and video together.
GPT-4V and Gemini can analyze charts, describe images, and answer visual questions.
The combination of multimodal perception, reasoning, tool use, and long-context
understanding is rapidly closing the gap between AI and human capabilities.

Fine-tuning allows adapting pre-trained models to specific domains with relatively
little data. LoRA and QLoRA make fine-tuning possible on consumer hardware.
PEFT techniques reduce the number of trainable parameters dramatically.
These advances democratize the ability to customize foundation models.

Evaluation remains one of the hardest problems in AI. Traditional metrics like
BLEU and ROUGE fail to capture semantic quality. LLM-as-judge approaches use
one model to evaluate another. RAGAS provides metrics for RAG pipelines specifically.
Human evaluation remains the gold standard but is expensive and slow.
""".strip()

# ─────────────────────────────────────────────────────────────
# Count tokens to understand why chunking might be needed
# cl100k_base is the encoding used by GPT-4 and many modern models.
# Ollama models use their own tokenizers, but this gives a good estimate.
# ─────────────────────────────────────────────────────────────
enc = tiktoken.get_encoding("cl100k_base")
token_count = len(enc.encode(SAMPLE_TEXT))

print(f"Sample text:        {len(SAMPLE_TEXT):,} characters")
print(f"Estimated tokens:   ~{token_count:,} tokens")
print(f"qwen2.5:7b limit:   ~32,768 tokens")
print(f"Fits in context?    {'✅ Yes' if token_count < 32768 else '❌ No — needs chunking'}")
print()
print("(Our sample fits — but a real 100-page PDF is ~50,000+ tokens and definitely needs chunking!)")

---
### Step 1: SPLIT — Chunking with Overlap

We break the document into smaller pieces called **chunks**. Each chunk must fit inside the model's context window.

**Why overlap?** If you cut exactly at character 1000, you might split a sentence in half:

```
Chunk 1: "...the transformer was introduced in 2017 in the paper Attention Is"
Chunk 2: "All You Need by Vaswani..."
```

Chunk 1 ends mid-sentence. The model summarizing chunk 1 will miss the point. Chunk 2 starts with a fragment that makes no sense alone.

**Overlap = repeat the last N characters of chunk 1 at the start of chunk 2.** This ensures no idea is cut off. A 10% overlap is usually enough.

```
chunk_size = 1000, overlap = 100

Chunk 1: chars   0 – 1000
Chunk 2: chars 900 – 1900   ← starts 100 chars before chunk 1 ends
Chunk 3: chars 1800 – 2800
```

In [ ]:
def split_into_chunks(text: str, chunk_size: int = 1000, overlap: int = 100) -> list:
    """
    Split a long text into overlapping fixed-size chunks.

    Args:
        text:       The full document string.
        chunk_size: Maximum characters per chunk.
        overlap:    How many characters to repeat between consecutive chunks.
                    Prevents losing context at chunk boundaries.

    Returns:
        List of dicts, each with keys:
        index, text, start, end, char_count
    """
    chunks = []   # we'll build this list up one chunk at a time
    start = 0     # character index where the current chunk begins
    idx   = 0     # chunk counter (0-based)

    while start < len(text):
        # Figure out where this chunk ends
        end = min(start + chunk_size, len(text))

        # 👇 Don't cut mid-word — find the last space before 'end'
        #    rfind returns the index of the rightmost space in text[start:end]
        if end < len(text):
            last_space = text.rfind(" ", start, end)
            if last_space > start:   # only if we actually found a space
                end = last_space     # snap the boundary to the word edge

        chunk_text = text[start:end].strip()   # remove leading/trailing whitespace

        if chunk_text:   # skip empty chunks (can happen at the very end)
            chunks.append({
                "index":      idx,
                "text":       chunk_text,
                "start":      start,
                "end":        end,
                "char_count": len(chunk_text),
            })
            idx += 1

        # 👇 Advance start — subtract overlap so the next chunk re-reads the tail of this one
        #    If we've already reached the end of the document, stop the loop.
        start = end - overlap if end < len(text) else len(text)

    return chunks


# ── Test the splitter ──────────────────────────────────────────
chunks = split_into_chunks(SAMPLE_TEXT, chunk_size=800, overlap=100)
print(f"Split into {len(chunks)} chunks\n")

for c in chunks:
    preview = c["text"][:80].replace("\n", " ")  # single line preview
    print(f"Chunk {c['index']+1}: chars {c['start']:>4}–{c['end']:>4}  ({c['char_count']} chars)")
    print(f"  Preview: {preview}…")
    print()

---
### Step 2: MAP — Summarize Each Chunk

We send **each chunk individually** to the LLM with a prompt that says "summarize this passage".

Key design choices:
- **Low temperature (0.2)** — we want consistent, factual bullet points, not creative prose
- **max_tokens=300** — summaries should be much shorter than the original
- **System prompt** — tells the model to stick to what's written, not hallucinate extra facts

In [ ]:
def summarize_chunk(chunk_text: str, chunk_index: int, total_chunks: int) -> str:
    """
    Summarize a single chunk using the LLM.
    This is the core MAP operation — called once per chunk.

    Args:
        chunk_text:   The text of this chunk.
        chunk_index:  0-based position (used only for progress display).
        total_chunks: Total number of chunks (used for progress display).

    Returns:
        A short bullet-point summary as a string.
    """
    print(f"  Summarizing chunk {chunk_index+1}/{total_chunks}…", end=" ", flush=True)
    t0 = time.time()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise summarizer. Extract the key ideas from the "
                    "provided text into 2–4 bullet points. Be concise. "
                    "Only include what is explicitly stated — do not add outside knowledge."
                ),
            },
            {
                "role": "user",
                "content": f"Summarize this passage:\n\n{chunk_text}",
            },
        ],
        temperature=0.2,   # 👈 low = deterministic, consistent bullets
        max_tokens=300,    # 👈 keep summaries short
    )

    summary = response.choices[0].message.content   # extract the text from the response
    elapsed = round(time.time() - t0, 1)
    print(f"done ({elapsed}s)")
    return summary


# ── Test on just the first chunk so you can see what the output looks like ──
test_summary = summarize_chunk(chunks[0]["text"], 0, len(chunks))
print(f"\nChunk 1 summary:\n{test_summary}")

In [ ]:
# ── MAP all chunks sequentially ────────────────────────────────
# We loop through every chunk and call summarize_chunk() on each one.
# This is sequential (one at a time). Section 4 shows how to parallelize it.

print(f"MAP step: summarizing {len(chunks)} chunks…\n")
t_start = time.time()

chunk_summaries = []                       # will hold one summary string per chunk
for i, chunk in enumerate(chunks):
    summary = summarize_chunk(chunk["text"], i, len(chunks))
    chunk_summaries.append(summary)        # collect results in order

total_time = round(time.time() - t_start, 1)
print(f"\n✅ MAP complete: {len(chunk_summaries)} summaries in {total_time}s")
print(f"\nFirst chunk summary:\n{chunk_summaries[0]}")

---
### Step 3: REDUCE — Merge All Summaries into One

Now we have one short summary per chunk. We send **all of them together** to the LLM with a different prompt: "merge these into one coherent final summary".

Why does this work?
- Each individual summary is short (200–300 words)
- Even 10 chunk summaries = ~2,000–3,000 words total — well within any context window
- The LLM deduplicates, reorders, and connects the ideas into readable prose

This is why Map-Reduce is so powerful: **you never need to fit the whole document in context at once.**

In [ ]:
def merge_summaries(summaries: list) -> str:
    """
    Merge all chunk summaries into one final coherent summary.
    This is the REDUCE step.

    Args:
        summaries: List of summary strings, one per chunk.

    Returns:
        A single, well-structured final summary string.
    """
    # 👇 Label each section so the LLM knows they came from sequential parts of a document
    combined = "\n\n".join(
        f"--- Section {i+1} ---\n{s}"
        for i, s in enumerate(summaries)
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert synthesizer. Below are summaries of consecutive "
                    "sections of a longer document. Merge them into ONE coherent, "
                    "well-structured final summary. "
                    "Remove duplicates. Preserve all key ideas. Use clear paragraphs. "
                    "The final summary should read naturally — not as disconnected bullet points."
                ),
            },
            {
                "role": "user",
                "content": f"Merge these section summaries into one final summary:\n\n{combined}",
            },
        ],
        temperature=0.3,   # 👈 slightly higher than MAP — we want readable prose, not just bullets
        max_tokens=800,    # 👈 final summary can be longer than individual chunk summaries
    )

    return response.choices[0].message.content


# ── Run the REDUCE step ────────────────────────────────────────
print("REDUCE step: merging all summaries…\n")
t0 = time.time()
final_summary = merge_summaries(chunk_summaries)
elapsed = round(time.time() - t0, 1)

print(f"✅ Merge complete in {elapsed}s\n")
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(final_summary)
print("\n" + "=" * 60)

# 👇 Show the compression ratio — how much did we shrink the document?
original_chars = len(SAMPLE_TEXT)
summary_chars  = len(final_summary)
reduction_pct  = round((1 - summary_chars / original_chars) * 100)
print(f"\nCompression: {original_chars:,} chars → {summary_chars:,} chars ({reduction_pct}% reduction)")

---
## 🛠️ SECTION 2 — GUIDED BUILD: Wrap It in a Class

So far we've written three standalone functions: `split_into_chunks`, `summarize_chunk`, and `merge_summaries`.

Real software bundles related functions into a **class** so you can:
- Store configuration (model name, chunk size) in one place
- Call `.summarize(text)` with a single line
- Swap configurations easily (test with chunk_size=500, deploy with chunk_size=2000)

Your job: fill in every `___` placeholder below.

**Rules:**
- Replace `___` with the correct value
- Do NOT change any other code
- Run the test cell at the bottom to check your answers

**Hints are in the comments above each `___`.**

In [ ]:
class MapReduceSummarizer:
    """
    A self-contained Map-Reduce summarization pipeline.

    Usage:
        summarizer = MapReduceSummarizer(client, model="qwen2.5:7b")
        result = summarizer.summarize(long_text)
        print(result["final_summary"])
    """

    def __init__(self, client, model: str = "qwen2.5:7b",
                 chunk_size: int = 1000, overlap: int = 100):
        # 👇 Store all settings on self so every method can access them
        self.client     = client
        self.model      = model
        self.chunk_size = chunk_size
        self.overlap    = overlap

    # ── SPLIT ─────────────────────────────────────────────────
    def split(self, text: str) -> list:
        """Split text into overlapping chunks."""
        # Hint: call split_into_chunks() with text, self.chunk_size, self.overlap
        return split_into_chunks(___, self.chunk_size, self.overlap)  # TODO 1: pass text

    # ── MAP (single chunk) ────────────────────────────────────
    def map_one(self, chunk: dict) -> str:
        """Summarize a single chunk — the MAP operation applied to one element."""
        response = self.client.chat.completions.create(
            model=___,   # TODO 2: use self.model (don't hardcode the model name)
            messages=[
                {
                    "role": "system",
                    "content": "Summarize this passage into 2–4 bullet points. Be concise.",
                },
                {
                    "role": "user",
                    # Hint: the chunk text lives in chunk["text"]
                    "content": ___,  # TODO 3: use chunk["text"]
                },
            ],
            temperature=0.2,
            max_tokens=300,
        )
        # Hint: response.choices[0].message.content holds the LLM's reply
        return ___  # TODO 4: return the response text

    # ── MAP (all chunks) ──────────────────────────────────────
    def map_all(self, chunks: list) -> list:
        """Summarize every chunk. Returns a list of summary strings."""
        summaries = []
        for i, chunk in enumerate(chunks):
            print(f"  Summarizing chunk {i+1}/{len(chunks)}…")
            # Hint: call self.map_one() and pass the chunk dict
            summary = self.map_one(___)  # TODO 5: pass the chunk
            # Hint: add the summary string to the summaries list
            summaries.append(___)        # TODO 6: append the summary
        return summaries

    # ── REDUCE ────────────────────────────────────────────────
    def reduce(self, summaries: list) -> str:
        """Merge all summaries into the final output."""
        # Hint: call the standalone merge_summaries() function we wrote earlier
        return merge_summaries(___)  # TODO 7: pass summaries

    # ── Full pipeline ─────────────────────────────────────────
    def summarize(self, text: str) -> dict:
        """
        Run the full Map-Reduce pipeline and return a results dict.

        Returns:
            {
                'final_summary':   str,
                'chunk_count':     int,
                'chunk_summaries': list[str],
                'original_chars':  int,
                'summary_chars':   int,
            }
        """
        print(f"Input: {len(text):,} chars")

        # Step 1 — Split
        # Hint: call self.split() and pass text
        chunks = self.split(___)   # TODO 8: pass text
        print(f"Split into {len(chunks)} chunks")

        # Step 2 — Map
        print("MAP step…")
        # Hint: call self.map_all() and pass chunks
        summaries = self.map_all(___)  # TODO 9: pass chunks

        # Step 3 — Reduce
        print("REDUCE step…")
        # Hint: call self.reduce() and pass summaries
        final = self.reduce(___)   # TODO 10: pass summaries

        return {
            "final_summary":   final,
            "chunk_count":     len(chunks),
            "chunk_summaries": summaries,
            "original_chars":  len(text),
            "summary_chars":   len(final),
        }


# ── Test your implementation ───────────────────────────────────
# If all TODOs are correct, this should print a final summary.
summarizer = MapReduceSummarizer(client, model=MODEL, chunk_size=800, overlap=100)
result = summarizer.summarize(SAMPLE_TEXT)

print("\n" + "=" * 60)
print("FINAL SUMMARY (from your class)")
print("=" * 60)
print(result["final_summary"])
print(f"\nCompression: {result['original_chars']:,} chars → {result['summary_chars']:,} chars")

---
## 🔬 SECTION 3 — EXPERIMENT: What Happens When You Change the Parameters?

The best way to understand a system is to **break it deliberately**. Each cell below changes one variable while keeping everything else the same. Read the observation note at the bottom of each cell.

You don't need to edit any code here — just **run each cell and observe the output**.

In [ ]:
# ── EXPERIMENT 1: How does chunk_size affect the result? ───────
#
# We test three chunk sizes on the same document.
# Observe: number of chunks, average chunk size.
# Think: more chunks = more LLM calls = slower & more expensive
#        fewer chunks = faster but each chunk has more content to summarize

results = {}

for chunk_size in [300, 800, 2000]:
    chunks = split_into_chunks(SAMPLE_TEXT, chunk_size=chunk_size, overlap=100)
    avg_chars = sum(c["char_count"] for c in chunks) // len(chunks)
    print(f"chunk_size={chunk_size:>4}:  {len(chunks)} chunks,  avg {avg_chars} chars/chunk")
    results[chunk_size] = len(chunks)

print()
print("📝 Observation:")
print("  - Smaller chunks → more LLM calls (MAP step is slower and costs more)")
print("  - Larger chunks → fewer calls, but each summary needs to cover more content")
print("  - Rule of thumb: chunk_size ≈ 20–30% of your model's context window")
print("  - For qwen2.5:7b (32k context), chunk_size=1000–2000 chars is a good starting point")

In [ ]:
# ── EXPERIMENT 2: What does overlap actually do? ───────────────
#
# We use a short text that has a sentence spanning a chunk boundary.
# Compare overlap=0 (boundary cut) vs overlap=50 (boundary softened).

boundary_text = (
    "The transformer architecture was introduced in 2017 in the paper "
    "Attention Is All You Need by Vaswani et al. at Google Brain. "
    "It replaced recurrent connections with self-attention mechanisms. "
    "This was the key innovation that enabled modern large language models."
)

print("=== With overlap=0 (hard cut — context lost at boundaries) ===")
chunks_no_overlap = split_into_chunks(boundary_text, chunk_size=120, overlap=0)
for c in chunks_no_overlap:
    print(f"  Chunk {c['index']+1}: '{c['text']}'")

print()
print("=== With overlap=40 (soft boundary — tail of chunk N repeated in chunk N+1) ===")
chunks_overlap = split_into_chunks(boundary_text, chunk_size=120, overlap=40)
for c in chunks_overlap:
    print(f"  Chunk {c['index']+1}: '{c['text']}'")

print()
print("📝 Observation:")
print("  - With overlap=0, chunk 1 may end mid-idea and chunk 2 starts without context")
print("  - With overlap=40, chunk 2 repeats the last ~40 chars of chunk 1 as context")
print("  - This ensures the LLM summarizing chunk 2 understands what came before")
print("  - Trade-off: overlap duplicates some text, slightly increasing total tokens processed")

In [ ]:
# ── EXPERIMENT 3: Measure the time breakdown of each step ──────
#
# Running the full pipeline and timing each phase shows you where the
# time actually goes. Spoiler: SPLIT is instant. MAP is the bottleneck.

print(f"Input: {len(SAMPLE_TEXT):,} chars\n")

# ── SPLIT timing ──────────────────────────────────────────────
t0 = time.time()
chunks = split_into_chunks(SAMPLE_TEXT, chunk_size=800, overlap=100)
split_time = round(time.time() - t0, 4)
print(f"SPLIT:  {len(chunks)} chunks in {split_time}s  ← no LLM, just string slicing")

# ── MAP timing ────────────────────────────────────────────────
t0 = time.time()
summaries = []
for i, c in enumerate(chunks):
    s = summarize_chunk(c["text"], i, len(chunks))
    summaries.append(s)
map_time = round(time.time() - t0, 1)
avg_per_chunk = round(map_time / len(chunks), 1)
print(f"MAP:    {len(chunks)} LLM calls in {map_time}s  ({avg_per_chunk}s/chunk avg)")

# ── REDUCE timing ─────────────────────────────────────────────
t0 = time.time()
final = merge_summaries(summaries)
reduce_time = round(time.time() - t0, 1)
print(f"REDUCE: 1 LLM call in {reduce_time}s")

# ── Results summary ───────────────────────────────────────────
total = round(split_time + map_time + reduce_time, 1)
compression = round((1 - len(final) / len(SAMPLE_TEXT)) * 100)

print(f"\n📊 PIPELINE RESULTS:")
print(f"  Original:    {len(SAMPLE_TEXT):,} chars")
print(f"  Summary:     {len(final):,} chars")
print(f"  Compression: {compression}%")
print(f"  Total time:  {total}s")
print()
print("📝 Observation:")
print("  - SPLIT is O(n) in document length — effectively instant")
print("  - MAP dominates total time (N LLM calls)")
print("  - Parallelizing MAP is the biggest performance win (see Section 4 Challenge 1)")

---
## 🏆 SECTION 4 — CHALLENGE

Three challenges. **Hints are given — answers are not.** Work through them in order; each builds on the last.

---

### Challenge 1: Parallel MAP ⚡

**Problem:** The current MAP step runs chunks one at a time. If you have 10 chunks and each takes 3 seconds, the MAP step takes 30 seconds.

**Your task:** Make MAP parallel — send all chunk requests at the same time using threads. With 3 workers and 10 chunks, 3 chunks run simultaneously, cutting time to ~10 seconds.

**Hints:**
- Use `concurrent.futures.ThreadPoolExecutor`
- Each future should return `(index, summary)` — a tuple — so you can reassemble results in the original order
- `as_completed(futures)` yields futures as they finish — **not necessarily in the order you submitted them** — so you need the index

```python
from concurrent.futures import ThreadPoolExecutor, as_completed
```

---

### Challenge 2: Hierarchical Reduce 🌲

**Problem:** If you have 50 chunks, the REDUCE step tries to merge 50 summaries at once. That combined text might itself exceed the context window!

**Your task:** Build a `hierarchical_reduce` that merges summaries in pairs:
- Round 1: merge [s1,s2]→m1, [s3,s4]→m2, [s5,s6]→m3, …
- Round 2: merge [m1,m2]→n1, [m3]→n2, …
- … repeat until 1 summary remains

**Hints:**
- Think recursively: if `len(summaries) == 1`, return `summaries[0]` (base case)
- Otherwise, loop in steps of 2, call `merge_summaries([s1, s2])` for each pair, collect results, recurse
- Handle odd-length lists: the last summary has no pair — carry it forward unchanged

---

### Challenge 3: Styled Summarization 🎨

**Problem:** Different audiences need different summaries. A CEO wants 3 bullet points. A researcher wants technical depth. A junior wants plain English.

**Your task:** Add a `style` parameter to `MapReduceSummarizer` that accepts `"bullets"`, `"executive"`, or `"technical"`. Each style uses a different system prompt in both MAP and REDUCE.

**Hints:**
- Store prompts in a dict: `STYLE_PROMPTS = {"bullets": {...}, "executive": {...}, "technical": {...}}`
- Pass `style` to `__init__` and store it as `self.style`
- In `map_one` and `reduce`, look up the correct prompt with `STYLE_PROMPTS[self.style]`
- Test: run the same text through all three styles and compare

In [ ]:
# ── Challenge 1 scaffold: Parallel MAP ────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed

def map_all_parallel(chunks: list, max_workers: int = 3) -> list:
    """
    Parallel MAP: summarize all chunks concurrently using a thread pool.
    max_workers=3 means up to 3 LLM calls happen at the same time.

    Args:
        chunks:      List of chunk dicts (from split_into_chunks).
        max_workers: Number of parallel threads.

    Returns:
        List of summary strings, in the SAME order as chunks (index 0 = chunk 0).
    """
    # Pre-allocate the result list so we can store by index regardless of completion order
    summaries = [None] * len(chunks)

    def _one_chunk(i: int, chunk: dict):
        """
        Summarize chunk i and return (i, summary) so we can store at the right index.
        """
        # Hint: call summarize_chunk() with chunk["text"], i, len(chunks)
        summary = summarize_chunk(___, i, len(chunks))  # TODO A: pass chunk["text"]
        return i, ___                                   # TODO B: return (i, summary)

    with ThreadPoolExecutor(max_workers=___) as executor:   # TODO C: use max_workers
        # Submit one future per chunk
        futures = {
            executor.submit(_one_chunk, i, chunk): i
            for i, chunk in enumerate(___)   # TODO D: iterate over chunks
        }

        # Collect results as they complete (order varies!)
        for future in as_completed(futures):
            i, summary = future.result()     # unpack the (index, summary) tuple
            summaries[___] = summary         # TODO E: store at the correct index

    return summaries


# ── Test: compare timing between sequential and parallel ───────
test_chunks = split_into_chunks(SAMPLE_TEXT, chunk_size=800, overlap=100)

print("Parallel MAP:")
t0 = time.time()
parallel_summaries = map_all_parallel(test_chunks, max_workers=3)
parallel_time = round(time.time() - t0, 1)
print(f"\n✅ Done in {parallel_time}s  ({len(parallel_summaries)} summaries)")
print("\nFirst parallel summary:")
print(parallel_summaries[0])